In [0]:
dbutils.widgets.text("sas_token", "", "Azure Storage SAS Token")
dbutils.widgets.text("storage_account", "stdataengprojectrafael", "Storage Account")
dbutils.widgets.text("target_schemas", "silver,gold", "Schemas to Export (comma-separated)")

In [0]:
import io
import os
import requests
import pandas as pd

def get_sas_token() -> str:
    token = os.getenv("AZURE_SAS_TOKEN", "").strip() or dbutils.widgets.get("sas_token").strip()
    if not token:
        raise ValueError("Fill in the 'Azure Storage SAS Token' widget at the top of the notebook.")
    return token.lstrip("?")

storage_account = dbutils.widgets.get("storage_account").strip()
schemas = [s.strip() for s in dbutils.widgets.get("target_schemas").split(",") if s.strip()]
sas_token = get_sas_token()

def upload_table_as_parquet(full_table_name: str, container: str, blob_path: str):
    df_spark = spark.read.table(full_table_name)

    rows = [row.asDict(recursive=True) for row in df_spark.collect()]

    if not rows:
        raise ValueError(f"Table is empty: {full_table_name}")

    df_pd = pd.DataFrame(rows, columns=df_spark.columns)

    buffer = io.BytesIO()
    df_pd.to_parquet(buffer, index=False)
    buffer.seek(0)

    url = (
        f"https://{storage_account}.blob.core.windows.net/"
        f"{container}/{blob_path}?{sas_token}"
    )

    headers = {
        "x-ms-blob-type": "BlockBlob",
        "Content-Type": "application/octet-stream"
    }

    resp = requests.put(
        url,
        headers=headers,
        data=buffer.getvalue()
    )
    resp.raise_for_status()

    print(
        f"Success: {full_table_name} "
        f"({len(df_pd)} lines) -> {container}/{blob_path}"
    )

for schema in schemas:
    tables_df = spark.sql(f"SHOW TABLES IN {schema}")
    for row in tables_df.collect():
        if row["isTemporary"]:
            continue
        
        table_name = row["tableName"]
        full_name = f"{schema}.{table_name}"
        clean_folder = table_name.replace("worldbank_", "")
        blob_path = f"world-bank/{clean_folder}/{table_name}.parquet"
        
        upload_table_as_parquet(full_name, container=schema, blob_path=blob_path)